## 개인 프롬프트를 LangSmith Prompt Hub 에 등록하기 (최신 권장 방식)

> 원본: CH02 `04-Personal-Prompts.ipynb`  
> 기준: LangChain 1.x / LangSmith SDK (2026년 9월)

### 🔄 변경 사항 (노트북 전체 공통)
- `from langchain import hub` → `from langsmith import Client` 후 `client.push_prompt(...)`
- `from langchain.prompts import ...` → `from langchain_core.prompts import ...`
- `hub.push(f"{PROMPT_OWNER}/{prompt_title}", prompt)` → `client.push_prompt(prompt_title, object=prompt)`  
  owner 를 생략하면 API 키가 속한 **내 워크스페이스**에 저장됩니다. 그래서 `PROMPT_OWNER` 변수가 더 이상 필요 없습니다.
- `parent_commit_hash="latest"` 는 `push_prompt()` 의 **기본값**이므로 생략했습니다.
- 반복되는 import 는 맨 위에서 한 번만 합니다.
- 프롬프트 본문의 명백한 오타 몇 개를 바로잡았습니다. (`SUMMARY:"` 의 불필요한 따옴표, `semantic m   eaning`, `답변헤`)

In [ ]:
# %pip install -qU langsmith langchain-core python-dotenv

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # LANGSMITH_API_KEY 로드

In [ ]:
from langsmith import Client
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

client = Client()

## 요약: Stuff Documents

In [ ]:
prompt_title = "summary-stuff-documents"

# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt_template = """Please summarize the sentence according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. DO NOT include any unnecessary information.

CONTEXT:
{context}

SUMMARY:
"""
prompt = PromptTemplate.from_template(prompt_template)
prompt

In [ ]:
client.push_prompt(prompt_title, object=prompt)

## Map Prompt

In [ ]:
prompt_title = "map-prompt"

# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt_template = """You are a helpful expert journalist in extracting the main themes from a GIVEN DOCUMENTS below.
Please provide a comprehensive summary of the GIVEN DOCUMENTS in numbered list format. 
The summary should cover all the key points and main ideas presented in the original text, while also condensing the information into a concise and easy-to-understand format. 
Please ensure that the summary includes relevant details and examples that support the main ideas, while avoiding any unnecessary information or repetition. 
The length of the summary should be appropriate for the length and complexity of the original text, providing a clear and accurate overview without omitting any important information.

GIVEN DOCUMENTS:
{docs}

FORMAT:
1. main theme 1
2. main theme 2
3. main theme 3
...

CAUTION:
- DO NOT list more than 5 main themes.

Helpful Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)
prompt

In [ ]:
client.push_prompt(prompt_title, object=prompt)

## Reduce Prompt

In [ ]:
prompt_title = "reduce-prompt"

# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt_template = """You are a helpful expert in summary writing.
You are given numbered lists of summaries.
Extract top 10 most important insights from the summaries.
Then, write a summary of the insights in KOREAN.

LIST OF SUMMARIES:
{doc_summaries}

Helpful Answer:
"""
prompt = PromptTemplate.from_template(prompt_template)
prompt

In [ ]:
client.push_prompt(prompt_title, object=prompt)

In [ ]:
prompt_title = "chain-of-density-reduce-korean"

# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt_template = """You are a helpful expert in summary writing. You are given lists of summaries.
Please sum up previously summarized sentences according to the following REQUEST.
REQUEST:
1. Summarize the main points in bullet points in KOREAN.
2. Each summarized sentence must start with an emoji that fits the meaning of the each sentence.
3. Use various emojis to make the summary more interesting.
4. MOST IMPORTANT points should be organized at the top of the list.
5. DO NOT include any unnecessary information.

LIST OF SUMMARIES:
{doc_summaries}

Helpful Answer: """
prompt = PromptTemplate.from_template(prompt_template)
prompt

In [ ]:
client.push_prompt(prompt_title, object=prompt)

## Metadata Tagger

> 💡 참고: 이 프롬프트처럼 "JSON 형식으로 답하라"고 지시하는 방식은 지금은 **구조화 출력**(`llm.with_structured_output(PydanticModel)`)으로 대체하는 것이 권장됩니다. 스키마까지 Hub 에 함께 저장하고 싶다면 `langchain_core.prompts.structured.StructuredPrompt` 를 `push_prompt()` 로 올릴 수 있습니다. 여기서는 원본 학습 흐름대로 프롬프트만 등록합니다.

In [ ]:
prompt_title = "metadata-tagger"

# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt_template = """Given the following product review, conduct a comprehensive analysis to extract key aspects mentioned by the customer, with a focus on evaluating the product's design and distinguishing between positive aspects and areas for improvement. 
Identify primary features or attributes of the product that the customer appreciated or highlighted, specifically looking for mentions related to the feel of the keys, sound produced by the keys, overall user experience, charging aspect, and the design of the product, etc. 
Assess the overall tone of the review (positive, neutral, or negative) based on the sentiment expressed about these attributes. 
Additionally, provide a detailed evaluation of the design, outline the positive aspects that the customer enjoyed, and note any areas of improvement or disappointment mentioned. 
Extract the customer's rating of the product on a scale of 1 to 5, as indicated at the beginning of the review. 
Summarize your findings in a structured JSON format, including an array of keywords, evaluations for design, satisfaction points, improvement areas, the assessed tone, and the numerical rating. 

INPUT:
{input}

"""
prompt = ChatPromptTemplate.from_template(prompt_template)
prompt

In [ ]:
client.push_prompt(prompt_title, object=prompt)

## Chain of Density 요약

In [ ]:
prompt_title = "chain-of-density-korean"

# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt = ChatPromptTemplate.from_template(
    """Article: {ARTICLE}
You will generate increasingly concise, entity-dense summaries of the above article. 

Repeat the following 2 steps 5 times. 

Step 1. Identify 1-3 informative entities (";" delimited) from the article which are missing from the previously generated summary. 
Step 2. Write a new, denser summary of identical length which covers every entity and detail from the previous summary plus the missing entities. 

A missing entity is:
- relevant to the main story, 
- specific yet concise (100 words or fewer), 
- novel (not in the previous summary), 
- faithful (present in the article), 
- anywhere (can be located anywhere in the article).

Guidelines:

- The first summary should be long (8-10 sentences, ~200 words) yet highly non-specific, containing little information beyond the entities marked as missing. Use overly verbose language and fillers (e.g., "this article discusses") to reach ~200 words.
- Make every word count: rewrite the previous summary to improve flow and make space for additional entities.
- Make space with fusion, compression, and removal of uninformative phrases like "the article discusses".
- The summaries should become highly dense and concise yet self-contained, i.e., easily understood without the article. 
- Missing entities can appear anywhere in the new summary.
- Never drop entities from the previous summary. If space cannot be made, add fewer new entities. 

Remember, use the exact same number of words for each summary.
Answer in JSON. The JSON should be a list (length 5) of dictionaries whose keys are "Missing_Entities" and "Denser_Summary".
Use only KOREAN language to reply."""
)

In [ ]:
client.push_prompt(prompt_title, object=prompt)

## Chain of Density (Korean) - 2

In [ ]:
prompt_title = "chain-of-density-map-korean"

# 요약문을 작성하기 위한 프롬프트 정의 (직접 프롬프트를 작성하는 경우)
prompt = ChatPromptTemplate.from_template(
    """Article: {ARTICLE}
You will generate increasingly concise, entity-dense summaries of the above article. 

Repeat the following 2 steps 3 times. 

Step 1. Identify 1-3 informative entities (";" delimited) from the article which are missing from the previously generated summary. 
Step 2. Write a new, denser summary of identical length which covers every entity and detail from the previous summary plus the missing entities. 

A missing entity is:
- relevant to the main story, 
- specific yet concise (100 words or fewer), 
- novel (not in the previous summary), 
- faithful (present in the article), 
- anywhere (can be located anywhere in the article).

Guidelines:

- The first summary should be long (8-10 sentences, ~200 words) yet highly non-specific, containing little information beyond the entities marked as missing. Use overly verbose language and fillers (e.g., "this article discusses") to reach ~200 words.
- Make every word count: rewrite the previous summary to improve flow and make space for additional entities.
- Make space with fusion, compression, and removal of uninformative phrases like "the article discusses".
- The summaries should become highly dense and concise yet self-contained, i.e., easily understood without the article. 
- Missing entities can appear anywhere in the new summary.
- Never drop entities from the previous summary. If space cannot be made, add fewer new entities. 

Remember, use the exact same number of words for each summary.
Answer "Missing Entities" and "Denser_Summary" as in TEXT format.
Use only KOREAN language to reply."""
)

In [ ]:
client.push_prompt(prompt_title, object=prompt)

## RAG 문서 프롬프트

In [ ]:
prompt_title = "rag-korean"

In [ ]:
system = """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요. 답변은 한글로 답변해 주세요."""

human = """#Question: 
{question} 

#Context: 
{context} 

#Answer:"""

prompt = ChatPromptTemplate([("system", system), ("human", human)])

In [ ]:
# parent_commit_hash="latest" 는 기본값이므로 생략
client.push_prompt(prompt_title, object=prompt)

## RAG + 출처

In [ ]:
prompt_title = "rag-korean-with-source"

In [ ]:
system = """당신은 질문-답변(Question-Answering)을 수행하는 친절한 AI 어시스턴트입니다. 당신의 임무는 주어진 문맥(context) 에서 주어진 질문(question) 에 답하는 것입니다.
검색된 다음 문맥(context) 을 사용하여 질문(question) 에 답하세요. 만약, 주어진 문맥(context) 에서 답을 찾을 수 없다면, 답을 모른다면 `주어진 정보에서 질문에 대한 정보를 찾을 수 없습니다` 라고 답하세요.
기술적인 용어나 이름은 번역하지 않고 그대로 사용해 주세요. 출처(page, source)를 답변에 포함하세요. 답변은 한글로 답변해 주세요."""

human = """#Question: 
{question} 

#Context: 
{context} 

#Answer:"""

prompt = ChatPromptTemplate([("system", system), ("human", human)])

In [ ]:
client.push_prompt(prompt_title, object=prompt)

## LLM Evaluation

In [ ]:
prompt = PromptTemplate.from_template(
    """
As an LLM evaluator (judge), please assess the LLM's response to the given question. Evaluate the response's accuracy, comprehensiveness, and context precision based on the provided context. After your evaluation, return only the numerical scores in the following format:
Accuracy: [score]
Comprehensiveness: [score]
Context Precision: [score]
Final: [normalized score]
Grading rubric:

Accuracy (0-10 points):
Evaluate how well the answer aligns with the information provided in the given context.

0 points: The answer is completely inaccurate or contradicts the provided context
4 points: The answer partially aligns with the context but contains significant inaccuracies
7 points: The answer mostly aligns with the context but has minor inaccuracies or omissions
10 points: The answer fully aligns with the provided context and is completely accurate


Comprehensiveness (0-10 points):

0 points: The answer is completely inadequate or irrelevant
3 points: The answer is accurate but too brief to fully address the question
7 points: The answer covers main aspects but lacks detail or misses minor points
10 points: The answer comprehensively covers all aspects of the question


Context Precision (0-10 points):
Evaluate how precisely the answer uses the information from the provided context.

0 points: The answer doesn't use any information from the context or uses it entirely incorrectly
4 points: The answer uses some information from the context but with significant misinterpretations
7 points: The answer uses most of the relevant context information correctly but with minor misinterpretations
10 points: The answer precisely and correctly uses all relevant information from the context


Final Normalized Score:
Calculate by summing the scores for accuracy, comprehensiveness, and context precision, then dividing by 30 to get a score between 0 and 1.
Formula: (Accuracy + Comprehensiveness + Context Precision) / 30

#Given question:
{question}

#LLM's response:
{answer}

#Provided context:
{context}

Please evaluate the LLM's response according to the criteria above. 

In your output, include only the numerical scores for FINAL NORMALIZED SCORE without any additional explanation or reasoning.
ex) 0.81

#Final Normalized Score(Just the number):

"""
)

client.push_prompt("context-answer-evaluator", object=prompt)

In [ ]:
# 프롬프트
system = """You are a grader assessing relevance of a retrieved document to a user question. \n 
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."""

grade_prompt = ChatPromptTemplate(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {context} \n\n User question: {question}"),
    ]
)

client.push_prompt("retrieval-question-grader", object=grade_prompt)

In [ ]:
# Prompt
system = """You are a grader assessing relevance of a retrieved document to the answer. \n 
    It does not need to be a stringent test. The goal is to filter out erroneous retrievals. \n
    If the document contains keyword(s) or semantic meaning related to the answer, grade it as relevant. \n
    Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the answer."""

grade_prompt = ChatPromptTemplate(
    [
        ("system", system),
        ("human", "Retrieved document: \n\n {context} \n\n Answer: {answer}"),
    ]
)

client.push_prompt("retrieval-answer-grader", object=grade_prompt)

## (참고) 등록한 프롬프트 불러와서 사용하기

등록한 프롬프트는 이름만으로 가져올 수 있고, 바로 체인에 연결할 수 있습니다.

In [ ]:
rag_prompt = client.pull_prompt("rag-korean")
print(rag_prompt.invoke({"question": "테스트 질문", "context": "테스트 문맥"}).to_string())